# De píxeles a patrones: histogramas, filtros y aprendizaje

Una imagen puede representarse con números, resumirse mediante un histograma y analizarse con filtros que responden a patrones locales. El aprendizaje por gradientes permite ajustar parámetros para reducir un error.

Este ejemplo usa matrices pequeñas y Python sin librerías externas. Ilustra operaciones relacionadas con las redes convolucionales, sin implementar una red completa como LeNet-5.

## 1. Una imagen es una matriz

Esta matriz de 5×5 representa un siete. Cada elemento es un píxel: **0 indica negro y 1 indica blanco**. La función `mostrar` imprime los valores por filas; la representación con símbolos permite distinguir la figura.

In [1]:
imagen = [
    [1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0],
    [0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0],
]

def mostrar(matriz):
    for fila in matriz:
        print(" ".join(f"{valor:3}" for valor in fila))

mostrar(imagen)
print("\nEl mismo dibujo con símbolos: □ = 0, ■ = 1")
for fila in imagen:
    print(" ".join("■" if pixel else "□" for pixel in fila))

  1   1   1   1   1
  0   0   0   1   0
  0   0   1   0   0
  0   1   0   0   0
  0   1   0   0   0

El mismo dibujo con símbolos: □ = 0, ■ = 1
■ ■ ■ ■ ■
□ □ □ ■ □
□ □ ■ □ □
□ ■ □ □ □
□ ■ □ □ □


## 2. El histograma resume los colores

Un histograma cuenta cuántos píxeles tienen cada valor. Para esta imagen binaria, el resultado es una lista con las cantidades de negros y blancos.

Es como contar piezas LEGO por color: conocemos las cantidades, pero no la figura construida. Al invertir el orden de las filas, cambia la imagen y se conserva el histograma.

In [2]:
def histograma(imagen):
    conteo = [0, 0]  # Cantidades de píxeles negros y blancos.
    for fila in imagen:
        for pixel in fila:
            conteo[pixel] += 1
    return conteo

invertida = imagen[::-1]
mostrar(invertida)
print("\nOriginal  [negros, blancos]:", histograma(imagen))
print("Invertida [negros, blancos]:", histograma(invertida))
print("¿Son la misma matriz?", imagen == invertida)

  0   1   0   0   0
  0   1   0   0   0
  0   0   1   0   0
  0   0   0   1   0
  1   1   1   1   1

Original  [negros, blancos]: [16, 9]
Invertida [negros, blancos]: [16, 9]
¿Son la misma matriz? False


## 3. Un filtro compara regiones

Para examinar la distribución de los píxeles usamos un **kernel**, una pequeña matriz de pesos. Este filtro de 3×3 resta la columna izquierda, ignora el centro y suma la columna derecha. Así responde a cambios de intensidad asociados con bordes verticales.

La región siguiente es un ejemplo aislado de borde. Se multiplica cada píxel por el peso correspondiente y se suman los nueve productos.

In [3]:
region = [
    [0, 0, 1],
    [0, 0, 1],
    [0, 0, 1],
]

kernel = [
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
]

def aplicar_filtro(region, kernel):
    total = 0
    for i in range(3):
        for j in range(3):
            total += region[i][j] * kernel[i][j]
    return total

productos = [[region[i][j] * kernel[i][j] for j in range(3)] for i in range(3)]
print("Productos posición por posición:")
mostrar(productos)
print("Suma:", aplicar_filtro(region, kernel))

Productos posición por posición:
  0   0   1
  0   0   1
  0   0   1
Suma: 3


Cada fila aporta `0×(−1) + 0×0 + 1×1 = 1`; las tres filas suman **3**. El resultado mide la respuesta al contraste: no es una probabilidad ni una clasificación.

- **Positivo:** mayor intensidad en la columna derecha.
- **Negativo:** mayor intensidad en la columna izquierda.
- **Cero:** ambas columnas suman lo mismo, por uniformidad o cancelación.

Al invertir el borde cambia el signo; en una región uniforme la respuesta es cero.

In [4]:
uniforme = [[1, 1, 1], [1, 1, 1], [1, 1, 1]]
borde_contrario = [[1, 0, 0], [1, 0, 0], [1, 0, 0]]

print("Más blanco a la derecha:", aplicar_filtro(region, kernel))
print("Región uniforme:", aplicar_filtro(uniforme, kernel))
print("Más blanco a la izquierda:", aplicar_filtro(borde_contrario, kernel))

Más blanco a la derecha: 3
Región uniforme: 0
Más blanco a la izquierda: -3


## 4. Aplicar el filtro a toda la imagen

Desplazamos una ventana de 3×3 por la imagen del siete. En cada posición extraemos la región, multiplicamos por el mismo kernel y sumamos.

La primera ventana corresponde a la esquina superior izquierda de la imagen.

In [5]:
primera_ventana = [fila[:3] for fila in imagen[:3]]
mostrar(primera_ventana)
print("Primera respuesta:", aplicar_filtro(primera_ventana, kernel))

  1   1   1
  0   0   0
  0   0   1
Primera respuesta: 1


En esta ventana, la primera fila aporta `−1 + 0 + 1 = 0`, la segunda aporta `0` y la tercera aporta `1`. La primera casilla del resultado vale **1**.

La función `convolucion` repite la operación en todas las posiciones donde cabe la ventana. Los ciclos recorren filas y columnas; `ventana` extrae cada bloque y `aplicar_filtro` calcula su respuesta.

In [6]:
def convolucion(imagen, kernel):
    mapa = []
    for fila in range(len(imagen) - 2):
        fila_salida = []
        for columna in range(len(imagen[0]) - 2):
            ventana = [
                renglon[columna:columna + 3]
                for renglon in imagen[fila:fila + 3]
            ]
            fila_salida.append(aplicar_filtro(ventana, kernel))
        mapa.append(fila_salida)
    return mapa

mapa = convolucion(imagen, kernel)
print("Mapa de respuestas:")
mostrar(mapa)

Mapa de respuestas:
  1   1  -1
  1   0  -1
  1  -2  -1


El resultado es un **mapa de características de 3×3**. La ventana cabe en `5 − 3 + 1 = 3` posiciones por dimensión, avanzando un píxel por paso y sin añadir relleno.

Cada casilla representa la respuesta de una región de 3×3. El mapa conserva la ubicación de las respuestas, mientras que el histograma solo conserva cantidades. Este filtro detecta contrastes; por sí solo no reconoce el siete.

Técnicamente, la operación es correlación cruzada porque no se invierte el kernel. En las CNN es habitual llamarla convolución.

## 5. Aprendizaje por gradientes

El kernel anterior tiene valores elegidos manualmente. Para ilustrar cómo se aprende un parámetro, multiplicamos su respuesta por un **peso ajustable**.

Usamos el borde del paso 3, cuya respuesta es **3**, y fijamos una salida deseada de **1**. Ese objetivo es una etiqueta de juguete para el borde, no el número representado en la imagen.

| Operación | Fórmula |
|---|---|
| Predicción | `peso × entrada` |
| Error cuadrático | `(predicción − objetivo)²` |
| Gradiente respecto al peso | `2 × (predicción − objetivo) × entrada` |
| Actualización | `peso − paso × gradiente` |

El gradiente es la pendiente del error respecto al peso. Restar un pequeño múltiplo del gradiente permite buscar un peso con menor error. El tamaño del paso controla cuánto cambia el peso en cada actualización.

In [7]:
entrada = aplicar_filtro(region, kernel)
objetivo = 1
peso = 0.0
paso = 0.05

print("Iteración | Peso     | Predicción | Error")
for iteracion in range(5):
    prediccion = peso * entrada
    error = (prediccion - objetivo) ** 2
    print(f"{iteracion:9} | {peso:.6f} | {prediccion:.6f}   | {error:.8f}")
    gradiente = 2 * (prediccion - objetivo) * entrada
    peso -= paso * gradiente

Iteración | Peso     | Predicción | Error
        0 | 0.000000 | 0.000000   | 1.00000000
        1 | 0.300000 | 0.900000   | 0.01000000
        2 | 0.330000 | 0.990000   | 0.00010000
        3 | 0.333000 | 0.999000   | 0.00000100
        4 | 0.333300 | 0.999900   | 0.00000001


Con peso inicial **0**, la predicción es **0** y el error es **1**. El primer gradiente vale **−6**, de modo que el nuevo peso es `0 − 0.05×(−6) = 0.3`. Con ese peso, la predicción pasa a **0.9** y el error baja a **0.01**.

La tabla muestra los valores antes de cada actualización. Aquí se ajusta un único peso con un solo ejemplo; el kernel permanece fijo. Reducir ese error no demuestra que el modelo funcione con imágenes nuevas.

## Relación con el reconocimiento de documentos

En las redes convolucionales entrenadas para reconocer caracteres se ajustan muchos parámetros, incluidos los filtros, a partir de numerosos ejemplos. Los patrones locales se combinan en capas sucesivas para producir una clasificación.

Este notebook muestra tres ideas: **el histograma cuenta intensidades, el filtro examina patrones locales y el gradiente permite ajustar parámetros**. No incluye subsampling, una red completa ni clasificación de dígitos.

Referencia: LeCun, Bottou, Bengio y Haffner (1998), [Gradient-Based Learning Applied to Document Recognition](https://proceedingsoftheieee.ieee.org/gradient-based-learning-applied-to-document-recognition/).